In [1]:
import os

from blockymetamaterials.utils import SolutionData, EigenmodeData, ControlParams, GeometricalParams, LigamentParams, StretchingTorsionalSpringParams, MechanicalParams, save_data, load_data
from blockymetamaterials.geometry import RotatedSquareGeometry
from blockymetamaterials.analysis_utils import *

%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib import animation
from pathlib import Path
from tqdm import tqdm

import math

# import nlopt
import numpy as np
from numpy.linalg import norm
# import jax.numpy as jnp
# from jax import flatten_util
# from jax import random, grad, value_and_grad, jit, vmap, jacobian
from jax._src.config import config
config.update("jax_enable_x64", True)  # enable float64 type
# config.update("jax_log_compiles", 1)

# import scienceplots
# plt.style.use('science')

plt.rc('text', usetex=True)
plt.rcParams.update({'font.size': 14})
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams['animation.embed_limit'] = 2**128

from scipy.signal import argrelextrema
from sympy import symbols, lambdify, diff


No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


ModuleNotFoundError: No module named 'blockymetamaterials.analysis_utils'

# Local Functions

In [ ]:
exp_config = experimental_parameters()

print(f"Configuration: {exp_config['n1_blocks']}×{exp_config['n2_blocks']} lattice")
print(f"Total blocks: {exp_config['N']}")

def fit_conformal_polynomial(block_centroids, field, nc = 10, L = 1, n1_blocks = exp_config['n1_blocks'], n2_blocks = exp_config['n2_blocks']):
    # Complex reference coordinates with origin at lattice centre
    z_origin = np.array((block_centroids[0, 0] + 1j*block_centroids[0, 1])/2 + (block_centroids[n1_blocks*n2_blocks-1, 0] + 1j*block_centroids[n1_blocks*n2_blocks-1, 1]))/2
    zs = np.array((block_centroids[:, 0] + 1j*block_centroids[:, 1] - z_origin))

    u_comfields = field[:,0,:,0] + 1j*field[:,0,:,1]          # Convert to complex displacements

    coeffs, ufits, error_arr = conformal_fit(zs, L, u_comfields, nc, err_calc=True, frac = False)

    return coeffs, ufits, error_arr

def fitted_fields(fitfunc, block_centroids, n1_blocks = exp_config['n1_blocks'], n2_blocks = exp_config['n2_blocks']):
    # Complex reference coordinates with origin at lattice centre
    z_origin = np.array((block_centroids[0, 0] + 1j*block_centroids[0, 1])/2 + (block_centroids[n1_blocks*n2_blocks-1, 0] + 1j*block_centroids[n1_blocks*n2_blocks-1, 1]))/2
    zs = np.array((block_centroids[:, 0] + 1j*block_centroids[:, 1] - z_origin))

    if isinstance(fitfunc, np.ndarray) and callable(fitfunc[0]):
        complex_fields = np.array([fitfunc[frame](zs) for frame in range(len(fitfunc))])
        return np.stack((np.real(complex_fields), np.imag(complex_fields)), axis=-1)
    else:
        complex_field = fitfunc(zs)
        return np.stack((np.real(complex_field), np.imag(complex_field)), axis=-1)

def calculate_model_delta(field, error_sq, nc = 10, time_mask = None):
    """
    Args:
        field (np.ndarray): time-response data of shape (len(timepoints), n_blocks*2).
        error_sq (np.ndarray): error_sq in fitting conformal map to field, shape (len(timepoints),).
        nc (int): order of conformal polynomial map used for fitting and calculating errors.
        time_mask (np.ndarray, optional): to pick out relevant timepoints.

    Returns:
        model error (Delta) as a single number,
    """

    if time_mask is None:
        time_mask = np.ones((error_sq.shape[-1]), dtype = bool)

    field = np.where(np.isfinite(field), field, np.nan)
    field_var = field[time_mask] - np.nanmean(field[time_mask])

    residual_sum = np.sum(error_sq[time_mask])
    total_sum = np.nansum(field_var**2)

    return np.sqrt(residual_sum/total_sum)


# Experiment

In [ ]:
angle30 = 30.0 * np.pi/180.0

filepath = f'out/audrey/EXPERIMENTAL DATA/Neel_initial_angle_30.0/exp_response'

def load_exp_data(freq : int):
    block_centroids = np.load(f'{filepath}/{freq}Hz/block_centroids.npy')
    fields = np.load(f'{filepath}/{freq}Hz/fields.npy')
    timepoints = np.load(f'{filepath}/{freq}Hz/timepoints.npy')
    return block_centroids, fields, timepoints

exp_freqs = np.arange(2,36,1)

# Create mask for periodic regime: starting at timepoints = 0.064 and stopping at 0.064 + 10/freqs
# Relaxation regime starts at relax_pt = (0.064 + 10/freq)*1000 (index of timepoints array)
relax_pt = np.array([math.ceil((0.064 + 10/freq)*1000) for freq in exp_freqs])
exp_periodic_mask = []

for i, freq in enumerate(exp_freqs):
    exp_timepoints = np.load(f'out/audrey/EXPERIMENTAL DATA/Neel_initial_angle_30.0/exp_response/{freq}Hz/timepoints.npy')
    end_time = 0.064 + 10/freq
    exp_periodic_mask.append((exp_timepoints >= 0.064) & (exp_timepoints <= end_time))


In [ ]:
save_path = Path('notebooks/data/conformal_fitting_analysis/experiment_30_deg_sample')

nc = 10
L = (exp_config['n1_blocks']-1)*exp_config['spacing']

delta_arr = np.empty((len(exp_freqs),))

for i, freq in enumerate(exp_freqs):
    block_centroids, fields, timepoints = load_exp_data(freq)
    coeffs, ufits, error_arr = fit_conformal_polynomial(block_centroids, fields, nc=nc, L=L)

    print(coeffs.shape, error_arr.shape)

    fit_disp = fitted_fields(ufits, block_centroids)

    # Save results (ensure directories exist once per save target)
    coeff_dir = save_path / 'conformal_polynomial_coeffs_nc10'
    fitdisp_dir = save_path / 'conformal_polynomial_fittedfields_nc10'
    fiterror_dir = save_path / 'conformal_polynomial_fiterrors_nc10'
    for d in (coeff_dir, fitdisp_dir, fiterror_dir):
        d.mkdir(parents=True, exist_ok=True)
    
    # np.save(coeff_dir / f'{freq}Hz.npy', coeffs, allow_pickle=False)
    # np.save(fitdisp_dir / f'{freq}Hz.npy', fit_disp, allow_pickle=False)
    # np.save(fiterror_dir / f'{freq}Hz.npy', error_arr, allow_pickle=False)

    delta_arr[i] = calculate_model_delta(fields[:,0,:,:2].reshape((fields.shape[0],-1)), error_arr, nc=nc, time_mask=exp_periodic_mask[i])
    
    print(f"Finished processing frequency: {freq} Hz")
    

np.save(save_path / f'model_deltas_conformal_polynomial_fit_nc{nc}_periodicregime.npy', delta_arr, allow_pickle=False)


In [ ]:
angle5 = 5.0 * np.pi/180.0

filepath = f'out/audrey/EXPERIMENTAL DATA/Neel_initial_angle_5.0/exp_response'

def load_exp_data(freq : int):
    block_centroids = np.load(f'{filepath}/{freq}Hz/block_centroids.npy')
    fields = np.load(f'{filepath}/{freq}Hz/fields.npy')
    timepoints = np.load(f'{filepath}/{freq}Hz/timepoints.npy')
    return block_centroids, fields, timepoints

exp_freqs = np.arange(2,36,1)

# Create mask for periodic regime: starting at timepoints = 0.064 and stopping at 0.064 + 10/freqs
# Relaxation regime starts at relax_pt = (0.064 + 10/freq)*1000 (index of timepoints array)
relax_pt = np.array([math.ceil((0.064 + 10/freq)*1000) for freq in exp_freqs])
exp_periodic_mask = []

for i, freq in enumerate(exp_freqs):
    exp_timepoints = np.load(f'out/audrey/EXPERIMENTAL DATA/Neel_initial_angle_5.0/exp_response/{freq}Hz/timepoints.npy')
    end_time = 0.064 + 10/freq
    exp_periodic_mask.append((exp_timepoints >= 0.064) & (exp_timepoints <= end_time))


In [ ]:
save_path = Path('notebooks/data/conformal_fitting_analysis/experiment_5_deg_sample')

nc = 10
L = (exp_config['n1_blocks']-1)*exp_config['spacing']

delta_arr = np.empty((len(exp_freqs),))

for i, freq in enumerate(exp_freqs):
    block_centroids, fields, timepoints = load_exp_data(freq)
    coeffs, ufits, error_arr = fit_conformal_polynomial(block_centroids, fields, nc=nc, L=L)

    print(coeffs.shape, error_arr.shape)

    fit_disp = fitted_fields(ufits, block_centroids)

    # Save results (ensure directories exist once per save target)
    coeff_dir = save_path / 'conformal_polynomial_coeffs_nc10'
    fitdisp_dir = save_path / 'conformal_polynomial_fittedfields_nc10'
    fiterror_dir = save_path / 'conformal_polynomial_fiterrors_nc10'
    for d in (coeff_dir, fitdisp_dir, fiterror_dir):
        d.mkdir(parents=True, exist_ok=True)
    
    np.save(coeff_dir / f'{freq}Hz.npy', coeffs, allow_pickle=False)
    np.save(fitdisp_dir / f'{freq}Hz.npy', fit_disp, allow_pickle=False)
    np.save(fiterror_dir / f'{freq}Hz.npy', error_arr, allow_pickle=False)

    delta_arr[i] = calculate_model_delta(fields[:,0,:,:2].reshape((fields.shape[0],-1)), error_arr, nc=nc, time_mask=exp_periodic_mask[i])
    
    print(f"Finished processing frequency: {freq} Hz")
    

np.save(save_path / f'model_deltas_conformal_polynomial_fit_nc{nc}_periodicregime.npy', delta_arr, allow_pickle=False)

# Nonlinear Simulation

In [ ]:
angle30 = 30.0 * np.pi/180.0

# Setup geometry
geometry = RotatedSquareGeometry(
    n1_cells=exp_config['n1_blocks']//2, 
    n2_cells=exp_config['n2_blocks']//2, 
    spacing=exp_config['spacing'], 
    bond_length=exp_config['hinge_length']
)
block_centroids, centroid_node_vectors, bond_connectivity, reference_bond_vectors = geometry.get_parametrization()


filepath = f'out/audrey/FINAL EXP AND NONLIN DATASETS/simulation_fields/harmonic_input/30_deg_sample'

def load_nonlin_data(freq : int):
    fields = np.load(f'{filepath}/{freq}.0Hz/fields.npy')
    timepoints = np.load(f'{filepath}/{freq}.0Hz/10periods_sine_timepoints.npy')
    return fields, timepoints

# nonlin_freqs = exp_freqs
nonlin_freqs = np.arange(2,63,1)

# Create mask for periodic regime: starting at timepoints = 0.064 and stopping at 0.064 + 10/freqs
# Relaxation regime starts at relax_pt = (0.064 + 10/freq)*1000 (index of timepoints array)
relax_pt = np.array([math.ceil((0.064 + 10/freq)*1000) for freq in nonlin_freqs])
nonlin_periodic_mask = []

for i, freq in enumerate(nonlin_freqs):
    nonlin_timepoints = np.load(f'{filepath}/{freq}.0Hz/10periods_sine_timepoints.npy')
    end_time = 0.064 + 10/freq
    nonlin_periodic_mask.append((nonlin_timepoints >= 0.064) & (nonlin_timepoints <= end_time))


In [ ]:
save_path = Path('notebooks/data/conformal_fitting_analysis/nonlin_30_deg_sample_extended-freq-range')

nc = 10
L = (exp_config['n1_blocks']-1)*exp_config['spacing']

delta_arr = np.empty((len(nonlin_freqs),))

for i, freq in enumerate(nonlin_freqs):
    fields, timepoints = load_nonlin_data(freq)
    coeffs, ufits, error_arr = fit_conformal_polynomial(block_centroids(angle30), fields, nc=nc, L=L)

    fit_disp = fitted_fields(ufits, block_centroids(angle30))

    # Save results (ensure directories exist once per save target)
    coeff_dir = save_path / 'conformal_polynomial_coeffs_nc10'
    fitdisp_dir = save_path / 'conformal_polynomial_fittedfields_nc10'
    fiterror_dir = save_path / 'conformal_polynomial_fiterrors_nc10'
    for d in (coeff_dir, fitdisp_dir, fiterror_dir):
        d.mkdir(parents=True, exist_ok=True)
    
    # np.save(coeff_dir / f'{freq}Hz.npy', coeffs, allow_pickle=False)
    # np.save(fitdisp_dir / f'{freq}Hz.npy', fit_disp, allow_pickle=False)
    # np.save(fiterror_dir / f'{freq}Hz.npy', error_arr, allow_pickle=False)

    delta_arr[i] = calculate_model_delta(fields[:,0,:,:2].reshape((fields.shape[0],-1)), error_arr, nc=nc, time_mask=nonlin_periodic_mask[i])
    
    print(f"Finished processing frequency: {freq} Hz")
    

np.save(save_path / f'model_deltas_conformal_polynomial_fit_nc{nc}_periodicregime.npy', delta_arr, allow_pickle=False)


In [ ]:
angle5 = 5.0 * np.pi/180.0

# Setup geometry
geometry = RotatedSquareGeometry(
    n1_cells=exp_config['n1_blocks']//2, 
    n2_cells=exp_config['n2_blocks']//2, 
    spacing=exp_config['spacing'], 
    bond_length=exp_config['hinge_length']
)
block_centroids, centroid_node_vectors, bond_connectivity, reference_bond_vectors = geometry.get_parametrization()


filepath = f'out/audrey/FINAL EXP AND NONLIN DATASETS/simulation_fields/harmonic_input/5_deg_sample'

def load_nonlin_data(freq : int):
    fields = np.load(f'{filepath}/{freq}.0Hz/fields.npy')
    timepoints = np.load(f'{filepath}/{freq}.0Hz/10periods_sine_timepoints.npy')
    return fields, timepoints

# nonlin_freqs = exp_freqs
nonlin_freqs = np.arange(2,63,1)

# Create mask for periodic regime: starting at timepoints = 0.064 and stopping at 0.064 + 10/freqs
# Relaxation regime starts at relax_pt = (0.064 + 10/freq)*1000 (index of timepoints array)
relax_pt = np.array([math.ceil((0.064 + 10/freq)*1000) for freq in nonlin_freqs])
nonlin_periodic_mask = []

for i, freq in enumerate(nonlin_freqs):
    nonlin_timepoints = np.load(f'{filepath}/{freq}.0Hz/10periods_sine_timepoints.npy')
    end_time = 0.064 + 10/freq
    nonlin_periodic_mask.append((nonlin_timepoints >= 0.064) & (nonlin_timepoints <= end_time))

In [ ]:
save_path = Path('notebooks/data/conformal_fitting_analysis/nonlinear_5_deg_sample_extended-freq-range')

nc = 10
L = (exp_config['n1_blocks']-1)*exp_config['spacing']

delta_arr = np.empty((len(nonlin_freqs),))

for i, freq in enumerate(nonlin_freqs):
    fields, timepoints = load_nonlin_data(freq)
    coeffs, ufits, error_arr = fit_conformal_polynomial(block_centroids(angle5), fields, nc=nc, L=L)

    fit_disp = fitted_fields(ufits, block_centroids(angle5))

    # Save results (ensure directories exist once per save target)
    coeff_dir = save_path / 'conformal_polynomial_coeffs_nc10'
    fitdisp_dir = save_path / 'conformal_polynomial_fittedfields_nc10'
    fiterror_dir = save_path / 'conformal_polynomial_fiterrors_nc10'
    for d in (coeff_dir, fitdisp_dir, fiterror_dir):
        d.mkdir(parents=True, exist_ok=True)
    
    # np.save(coeff_dir / f'{freq}Hz.npy', coeffs, allow_pickle=False)
    # np.save(fitdisp_dir / f'{freq}Hz.npy', fit_disp, allow_pickle=False)
    # np.save(fiterror_dir / f'{freq}Hz.npy', error_arr, allow_pickle=False)

    delta_arr[i] = calculate_model_delta(fields[:,0,:,:2].reshape((fields.shape[0],-1)), error_arr, nc=nc, time_mask=nonlin_periodic_mask[i])
    
    print(f"Finished processing frequency: {freq} Hz")
    

np.save(save_path / f'model_deltas_conformal_polynomial_fit_nc{nc}_periodicregime.npy', delta_arr, allow_pickle=False)


# Linear Simulation - Eigenmodes

In [ ]:
# LOAD EIGENANALYSIS DATA

n1_blocks = exp_config['n1_blocks']
n2_blocks = exp_config['n2_blocks']
spacing = exp_config['spacing']

initial_angle = 30.0 * np.pi/180.0

linear_exp30_data = np.load('notebooks/data/linear_undamped_eigenmode_analysis/eigdata_24x16_initialangle_30.0_clamped.npz')
print(linear_exp30_data.keys())

frequencies = np.sqrt(linear_exp30_data['eigenvalues'])/(2*np.pi)
modes = linear_exp30_data['fields']
block_centroids = linear_exp30_data['block_centroids']

print(frequencies[:5])
print(modes.shape)

n_modes = 100

# Complex reference coordinates with origin at lattice centre
Lx = (n1_blocks-1)*spacing
z_origin = np.array((block_centroids[0, 0] + 1j*block_centroids[0, 1])/2 + (block_centroids[n1_blocks*n2_blocks-1, 0] + 1j*block_centroids[n1_blocks*n2_blocks-1, 1]))/2
zs = np.array((block_centroids[:, 0] + 1j*block_centroids[:, 1] - z_origin))

# Convert to complex displacements
u_comfields = modes[:n_modes,:,0] + 1j*modes[:n_modes,:,1]

In [ ]:
# Check fitting and error calculation for each mode, as a function of nc (order of conformal polynomial fit)
mode_range = np.arange(10)
nc_arr = np.arange(1,21)
error_arr = np.empty((len(nc_arr), len(mode_range)))

for j in range(len(nc_arr)):
    error_arr[j] = np.sqrt(conformal_fit(zs, Lx, u_comfields[mode_range], nc_arr[j], err_calc=True, frac = True)[2])

fig, axes = plt.subplots(figsize = (5.5, 3.5), tight_layout = True)

title = f'${n1_blocks} \\times {n2_blocks}$ | eigenmodes'

# colormap = cm.get_cmap('plasma', len(mode_range)*2)
# colorid = colormap(np.linspace(0.1, 0.65, len(mode_range)))

for m in range(10):
    axes.plot(nc_arr, error_arr[:,m], label=f'$u_{{{mode_range[m]+1}}}$')

axes.grid(which='major')

axes.set_yscale('log')
axes.set_ylim(0.05, 1.1)
axes.set_ylabel('$\Delta_{\mathrm{conf}}$')
axes.set_xlabel('$n_c$')
# axes.set_yticks([1e-4,1e-2,1])
axes.set_xlim(0.5, nc_arr[-1]+1)

axes.legend(loc='upper right', fontsize=8)

fig.suptitle(title)

In [ ]:
# Choose nc = 15 for fitting eigenmodes, as a balance between accuracy and overfitting (see plot above)

coeffs, ufits, error_sq = conformal_fit(zs, Lx, u_comfields[:n_modes], 15, err_calc=True, frac = True)
print(coeffs.shape, error_sq.shape)
error_nc15 = np.sqrt(error_sq)

# calculate fitted fields for nc=15
fitted_disp_nc15 = fitted_fields(ufits, block_centroids)

print("Errors for nc=15 conformal fit to first 10 eigenmodes:", error_nc15[:10])

# save as numpy array
np.save('notebooks/data/conformal_fitting_analysis/linear_eigenmodes_30_deg_sample/conformal_polynomial_fiterrors_nc15.npy', error_nc15, allow_pickle=False)
np.save('notebooks/data/conformal_fitting_analysis/linear_eigenmodes_30_deg_sample/conformal_polynomial_coeffs_nc15.npy', coeffs, allow_pickle=False)
np.save('notebooks/data/conformal_fitting_analysis/linear_eigenmodes_30_deg_sample/conformal_polynomial_fittedfields_nc15.npy', fitted_disp_nc15, allow_pickle=False)

# Linear Simulation - Better Eigenmodes

In [ ]:
# LOAD EIGENANALYSIS DATA

n1_blocks = 3*exp_config['n1_blocks']
n2_blocks = 3*exp_config['n2_blocks']
spacing = exp_config['spacing']

initial_angle = 30.0 * np.pi/180.0

linear_better30_data = np.load('notebooks/data/linear_undamped_eigenmode_analysis/eigdata_krotby100_72x48_initialangle_30.0_clamped.npz')
print(linear_better30_data.keys())

frequencies = np.sqrt(linear_better30_data['eigenvalues'])/(2*np.pi)
modes = linear_better30_data['fields']
block_centroids = linear_better30_data['block_centroids']

print(frequencies[:5])
print(modes.shape)

n_modes = 100

# Complex reference coordinates with origin at lattice centre
Lx = (n1_blocks-1)*spacing
z_origin = np.array((block_centroids[0, 0] + 1j*block_centroids[0, 1])/2 + (block_centroids[n1_blocks*n2_blocks-1, 0] + 1j*block_centroids[n1_blocks*n2_blocks-1, 1]))/2
zs = np.array((block_centroids[:, 0] + 1j*block_centroids[:, 1] - z_origin))

# Convert to complex displacements
u_comfields = modes[:n_modes,:,0] + 1j*modes[:n_modes,:,1]

In [ ]:
# Check fitting and error calculation for each mode, as a function of nc (order of conformal polynomial fit)
mode_range = np.arange(10)
nc_arr = np.arange(1,21)
error_arr = np.empty((len(nc_arr), len(mode_range)))

for j in range(len(nc_arr)):
    error_arr[j] = np.sqrt(conformal_fit(zs, Lx, u_comfields[mode_range], nc_arr[j], err_calc=True, frac = True)[2])

fig, axes = plt.subplots(figsize = (5.5, 3.5), tight_layout = True)

title = f'${n1_blocks} \\times {n2_blocks}$ | eigenmodes'

# colormap = cm.get_cmap('plasma', len(mode_range)*2)
# colorid = colormap(np.linspace(0.1, 0.65, len(mode_range)))

for m in range(10):
    axes.plot(nc_arr, error_arr[:,m], label=f'$u_{{{mode_range[m]+1}}}$')

axes.grid(which='major')

axes.set_yscale('log')
# axes.set_ylim(0.05, 1.1)
axes.set_ylabel('$\Delta_{\mathrm{conf}}$')
axes.set_xlabel('$n_c$')
# axes.set_yticks([1e-4,1e-2,1])
axes.set_xlim(0.5, nc_arr[-1]+1)

axes.legend(loc='upper right', fontsize=8)

fig.suptitle(title)

In [ ]:
# Choose nc = 15 for fitting eigenmodes, as a balance between accuracy and overfitting (see plot above)

coeffs, ufits, error_sq = conformal_fit(zs, Lx, u_comfields[:n_modes], 15, err_calc=True, frac = True)
print(coeffs.shape, error_sq.shape)
error_nc15 = np.sqrt(error_sq)

# calculate fitted fields for nc=15
fitted_disp_nc15 = fitted_fields(ufits, block_centroids)

print("Errors for nc=15 conformal fit to first 100 eigenmodes:", error_nc15[:10])

# save as numpy array
np.save('notebooks/data/conformal_fitting_analysis/linear_eigenmodes_30_deg_sample_krotby100_sizex3/conformal_polynomial_fiterrors_nc15.npy', error_nc15, allow_pickle=False)
np.save('notebooks/data/conformal_fitting_analysis/linear_eigenmodes_30_deg_sample_krotby100_sizex3/conformal_polynomial_coeffs_nc15.npy', coeffs, allow_pickle=False)
np.save('notebooks/data/conformal_fitting_analysis/linear_eigenmodes_30_deg_sample_krotby100_sizex3/conformal_polynomial_fittedfields_nc15.npy', fitted_disp_nc15, allow_pickle=False)

# Linear Response

In [ ]:
angle30 = 30.0 * np.pi/180.0

lin_freqs = 2+0.2*np.arange(300)

data = np.load('notebooks/data/linear_response_periodic_steadystate/24x16_initialangle_30.0_clamped_freqrange(2,60,0.2)Hz.npz')

block_centroids = data['block_centroids']
fields = data['fields']

n1_blocks = exp_config['n1_blocks']
n2_blocks = exp_config['n2_blocks']
spacing = exp_config['spacing']


In [ ]:
real_err = np.empty(len(lin_freqs))
imag_err = np.empty(len(lin_freqs))
total_err = np.empty(len(lin_freqs))
nc = 10
L = n1_blocks*spacing

z_origin = np.array((block_centroids[0, 0] + 1j*block_centroids[0, 1])/2 + (block_centroids[n1_blocks*n2_blocks-1, 0] + 1j*block_centroids[n1_blocks*n2_blocks-1, 1]))/2
zs = np.array(block_centroids[:, 0] + 1j*block_centroids[:, 1] - z_origin)

u_real = np.real(fields[:,:,0]) + 1j*np.real(fields[:,:,1])
u_imag = np.imag(fields[:,:,0]) + 1j*np.imag(fields[:,:,1])
u_mags = np.array([sum([abs(u_real[i,k])**2 + abs(u_imag[i,k])**2 for k in range(u_real.shape[1])]) for i in range(fields.shape[0])])

real_err = np.array(conformal_fit(zs, L, u_real, nc, err_calc=True, frac = False)[2])
imag_err = np.array(conformal_fit(zs, L, u_imag, nc, err_calc=True, frac = False)[2])

# u_mags = np.array([sum([abs(u_real[i,k])**2 + abs(u_imag[i,k])**2 for k in range(u_real.shape[1])]) for i in range(fields.shape[0])])
total_err = (real_err + imag_err)/ u_mags
print(u_mags[:5])

real_err = real_err / (np.array([norm(u_real[i]) for i in range(fields.shape[0])])**2)
imag_err = imag_err / (np.array([norm(u_imag[i]) for i in range(fields.shape[0])])**2)


In [ ]:
np.save('notebooks/data/conformal_fitting_analysis/linear_response_30_deg_sample_freqrange(2,60,0.2)Hz/model_deltas_conformal_polynomial_fit_nc10_periodicregime.npy', np.sqrt(total_err), allow_pickle=False)

In [ ]:
angle5 = 5.0 * np.pi/180.0

lin_freqs = 2+0.2*np.arange(300)

data = np.load('notebooks/data/linear_response_periodic_steadystate/24x16_initialangle_5.0_clamped_freqrange(2,60,0.2)Hz.npz')

block_centroids = data['block_centroids']
fields = data['fields']

n1_blocks = exp_config['n1_blocks']
n2_blocks = exp_config['n2_blocks']
spacing = exp_config['spacing']

In [ ]:
real_err = np.empty(len(lin_freqs))
imag_err = np.empty(len(lin_freqs))
total_err = np.empty(len(lin_freqs))
nc = 10
L = n1_blocks*spacing

z_origin = np.array((block_centroids[0, 0] + 1j*block_centroids[0, 1])/2 + (block_centroids[n1_blocks*n2_blocks-1, 0] + 1j*block_centroids[n1_blocks*n2_blocks-1, 1]))/2
zs = np.array(block_centroids[:, 0] + 1j*block_centroids[:, 1] - z_origin)

u_real = np.real(fields[:,:,0]) + 1j*np.real(fields[:,:,1])
u_imag = np.imag(fields[:,:,0]) + 1j*np.imag(fields[:,:,1])
u_mags = np.array([sum([abs(u_real[i,k])**2 + abs(u_imag[i,k])**2 for k in range(u_real.shape[1])]) for i in range(fields.shape[0])])

real_err = np.array(conformal_fit(zs, L, u_real, nc, err_calc=True, frac = False)[2])
imag_err = np.array(conformal_fit(zs, L, u_imag, nc, err_calc=True, frac = False)[2])

# u_mags = np.array([sum([abs(u_real[i,k])**2 + abs(u_imag[i,k])**2 for k in range(u_real.shape[1])]) for i in range(fields.shape[0])])
total_err = (real_err + imag_err)/ u_mags
print(u_mags[:5])

real_err = real_err / (np.array([norm(u_real[i]) for i in range(fields.shape[0])])**2)
imag_err = imag_err / (np.array([norm(u_imag[i]) for i in range(fields.shape[0])])**2)


In [ ]:
np.save('notebooks/data/conformal_fitting_analysis/linear_response_5_deg_sample_freqrange(2,60,0.2)Hz/model_deltas_conformal_polynomial_fit_nc10_periodicregime.npy', np.sqrt(total_err), allow_pickle=False)

# Combined Plot

In [ ]:
lin_freqs = 2+0.2*np.arange(300)
linsim30_err = np.load('notebooks/data/conformal_fitting_analysis/linear_response_30_deg_sample_freqrange(2,60,0.2)Hz/model_deltas_conformal_polynomial_fit_nc10_periodicregime.npy')
linsim5_err = np.load('notebooks/data/conformal_fitting_analysis/linear_response_5_deg_sample_freqrange(2,60,0.2)Hz/model_deltas_conformal_polynomial_fit_nc10_periodicregime.npy')

nonlin_freqs = np.arange(2,63,1)
delta_30sim = np.load('notebooks/data/conformal_fitting_analysis/nonlinear_30_deg_sample_extended-freq-range/model_deltas_conformal_polynomial_fit_nc10_periodicregime.npy')
delta_5sim = np.load('notebooks/data/conformal_fitting_analysis/nonlinear_5_deg_sample_extended-freq-range/model_deltas_conformal_polynomial_fit_nc10_periodicregime.npy')

exp_freqs = np.arange(2,36,1)
delta_30exp = np.load('notebooks/data/conformal_fitting_analysis/experiment_30_deg_sample/model_deltas_conformal_polynomial_fit_nc10_periodicregime.npy')
delta_5exp = np.load('notebooks/data/conformal_fitting_analysis/experiment_5_deg_sample/model_deltas_conformal_polynomial_fit_nc10_periodicregime.npy')

In [ ]:
col30 = 'blue'
col5 = 'red'

fig, ax1 = plt.subplots(figsize=(4,3.8), tight_layout = True)

linsim, = ax1.plot([],[], ':', color='k', label=r'linear sim')
nonlinsim, = ax1.plot([],[], '^', fillstyle='none', lw=1, markersize = 5, color='k', label=r'nonlinear sim')
exp, = ax1.plot([],[], 'o', markersize = 4.5, color='k', label=r'experiment')


# Initial angle = 30.0
line30, = ax1.plot([],[], 's', markersize=10, color=col30, label=r'$\theta_0=30^{\circ}$')

ax1.plot(lin_freqs, linsim30_err, ':', markersize = 4.5, color=col30)
ax1.plot(nonlin_freqs, delta_30sim, '^', fillstyle='none', lw=1, markersize = 5, color = col30)
ax1.plot(exp_freqs, delta_30exp, 'o', markersize = 4.5, color=col30)

# Initial angle = 5.0
line5, = ax1.plot([],[], 's', markersize=10, color=col5, label=r'$\theta_0=5^{\circ}$')

ax1.plot(lin_freqs, linsim5_err, ':', markersize = 4.5, color=col5)
ax1.plot(nonlin_freqs, delta_5sim, '^', fillstyle='none', lw=1, markersize = 5, color = col5)
ax1.plot(exp_freqs, delta_5exp, 'o', markersize = 4.5, color=col5)

fceil = 31.52420985648422
ceil = ax1.axvline(x=fceil, linestyle='-', color='g', lw=1, label=r'$f_{\mathrm{ceil}}$')

ax1.set_yscale('log')
ax1.set_xlabel('drive frequency $f_{\mathrm{d}}$ (Hz)')
ax1.set_ylabel(r'$\Delta_{\mathrm{conf}}$')

# ax1.set_yticks([0.1,1])
# ax1.set_yticklabels([r'$0.1$', r'$1$'])

ax1.set_xticks([0,10,20,30,40,fceil,50,60])
ax1.set_xticklabels([r'$0$',r'$10$',r'$20$','',r'$40$',r'$f_{\mathrm{ceil}}$',r'$50$',r'$60$'])
ax1.xaxis.get_ticklabels()[5].set_color('green')


# ax1.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
# ax1.set_title(rf'conformal fitting of response using {nc} terms | WITH TRANSLATIONS')

legend1 = ax1.legend(handles = [exp, nonlinsim, linsim], loc = 'upper left',
            fontsize=14, ncol=1, framealpha = 0, labelspacing=0.2, borderpad=0., handlelength=1.3, handletextpad=0.2)
ax1.legend(handles = [line30, line5], loc = 'lower right', bbox_to_anchor=(1.05,-0.03) , fontsize=14, ncol=1, framealpha = 0, labelspacing=0.2, handletextpad=0.1)

ax1.add_artist(legend1)

ax1.set_ylim(0.05,1.1)
ax1.set_xlim(None,63)



# Fit Animation for Experiments

In [ ]:
angle30 = 30.0 * np.pi/180.0

filepath = f'out/audrey/EXPERIMENTAL DATA/Neel_initial_angle_30.0/exp_response'

def load_exp_data(freq : int):
    block_centroids = np.load(f'{filepath}/{freq}Hz/block_centroids.npy')
    fields = np.load(f'{filepath}/{freq}Hz/fields.npy')
    timepoints = np.load(f'{filepath}/{freq}Hz/timepoints.npy')
    return block_centroids, fields, timepoints

exp_freqs = np.arange(2,36,1)

L = exp_config['n1_blocks']*10

n_tpts = 5201
nc = 10

i = 0
print(exp_freqs[i], 'Hz')

block_centroids, fields, timepoints = load_exp_data(exp_freqs[i])

z_origin = np.array((block_centroids[0, 0] + 1j*block_centroids[0, 1])/2 + (block_centroids[n1_blocks*n2_blocks-1, 0] + 1j*block_centroids[n1_blocks*n2_blocks-1, 1]))/2
zs = np.array((block_centroids[:, 0] + 1j*block_centroids[:, 1] - z_origin))

u_comfields = fields[:,0,:,0] + 1j*fields[:,0,:,1]

coeffs, ufits, error_arr = conformal_fit(zs, L, u_comfields, nc, err_calc=True, frac = False)

# Animate first 3 seconds of fitted fields for frequency index i
frames = 3000
com_disp = np.array([ufits[frame](zs) for frame in range(frames)])
realfit_disp = np.stack((np.real(com_disp), np.imag(com_disp)), axis=-1)

print(realfit_disp.shape)

In [ ]:
tstep = 5

n1_blocks = exp_config['n1_blocks']
n2_blocks = exp_config['n2_blocks']

clampedblockids = [0, 1, n1_blocks-2, n1_blocks-1, n1_blocks, 2*n1_blocks-1, (n2_blocks-1)*n1_blocks-1, n1_blocks*n2_blocks-2, n1_blocks*n2_blocks-1]
drivenblockids = [(n2_blocks-2)*n1_blocks, (n2_blocks-1)*n1_blocks, (n2_blocks-1)*n1_blocks+1]
col_arr =  ['blue']*len(block_centroids)
for id in clampedblockids:
    col_arr[id] = 'red'
for id in drivenblockids:
    col_arr[id] = 'green'

frames = np.arange(0,3000,tstep)

fig, axes = plt.subplots(figsize=(8,6), tight_layout = True)

axes.plot([],[])

def update(frame):
    axes.clear()
    for j in np.arange(n2_blocks):
        row_block_coordinates = np.array([block_centroids[n1_blocks*j:n1_blocks*(j+1), 0] + realfit_disp[frame, n1_blocks*j:n1_blocks*(j+1), 0],
                                            block_centroids[n1_blocks*j:n1_blocks*(j+1), 1] + realfit_disp[frame, n1_blocks*j:n1_blocks*(j+1), 1]])
        axes.plot(row_block_coordinates[0, :], row_block_coordinates[1, :], 'k', linewidth = 1)

    for k in np.arange(n1_blocks):
        col_block_coordinates = np.array([block_centroids[k:n1_blocks*(n2_blocks-1)+k+1:n1_blocks, 0] + realfit_disp[frame, k:n1_blocks*(n2_blocks-1)+k+1:n1_blocks, 0],
                                            block_centroids[k:n1_blocks*(n2_blocks-1)+k+1:n1_blocks, 1] + realfit_disp[frame, k:n1_blocks*(n2_blocks-1)+k+1:n1_blocks, 1]])
        axes.plot(col_block_coordinates[0, :], col_block_coordinates[1, :], 'k', linewidth = 1)
    
    cline, = axes.plot([],[],'k', label= 'fitted conformal mesh')

    exp_coords = block_centroids + fields[frame][0,:,:2]
    xs, ys = exp_coords.T
    axes.scatter(xs, ys, c = col_arr, s = 49, marker=r'$\odot$')
    rline, = axes.plot([],[],marker = r'$\odot$', linestyle = 'None', color = 'red', label = 'exp. clamped blocks')
    bline, = axes.plot([],[],marker = r'$\odot$', linestyle = 'None', color = 'blue', label = 'exp. free blocks')
    gline, = axes.plot([],[],marker = r'$\odot$', linestyle = 'None', color = 'green', label = 'exp. driven blocks')
    axes.set_axisbelow(True)
    axes.set_aspect('equal', adjustable='box')
    axes.set_title(f"Drive frequency: {exp_freqs[i]}Hz | time: {timepoints[frame]:.2f} sec")  
    axes.set_xlabel('x (mm)')
    axes.set_ylabel('y (mm)')
    axes.grid(False)
    axes.set_xlim(-20,250)
    axes.set_ylim(-20,200)
    axes.legend(loc = 'upper center', handles = [cline, bline, rline, gline], ncol = 2, fontsize = 16)
    return axes


ani = animation.FuncAnimation(fig=fig, func=update, frames=frames, blit = True, interval=50)
ani.save(str(f'notebooks/data/conformal_fit_animations/{exp_freqs[i]}Hz_exp_vs_conformalfit_{nc}termmodel_timestep_{tstep}.mp4'), writer="ffmpeg", dpi=200)
